This code shows the data pull from the FRED for 3-month Treasury bill both monthly and quarterly.


In [ ]:
import requests
import pandas as pd

# ----------------------------
# 1) SETTINGS
# ----------------------------
API_KEY = "b23d0fac17021282ceab7c2a02e11d9b"

START = "1960-01-01"
END   = "2026-02-28"

FRED_OBS_URL = "https://api.stlouisfed.org/fred/series/observations"

# OECD 3-month short-term interest rate (MEI) series IDs in FRED
# G7 + Spain
SERIES = {
    "United States": "IR3TIB01USM156N",
    "Canada":        "IR3TIB01CAM156N",
    "United Kingdom":"IR3TIB01GBM156N",
    "France":        "IR3TIB01FRM156N",
    "Germany":       "IR3TIB01DEM156N",
    "Italy":         "IR3TIB01ITM156N",
    "Japan":         "IR3TIB01JPM156N",
    "Spain":         "IR3TIB01ESM156N",
}

In [ ]:
# ----------------------------
# 2) FUNCTION TO DOWNLOAD ONE SERIES FROM FRED (USING REQUESTS)
# ----------------------------
def fetch_fred_series(series_id: str, start: str, end: str) -> pd.Series:
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": start,
        "observation_end": end,
    }
    r = requests.get(FRED_OBS_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    obs = data.get("observations", [])
    if not obs:
        raise ValueError(f"No observations returned for {series_id}")

    df = pd.DataFrame(obs)[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")  # '.' -> NaN
    df = df.set_index("date").sort_index()

    return df["value"]

In [ ]:
# ----------------------------
# 3) DOWNLOAD MONTHLY DATA FOR ALL COUNTRIES
# ----------------------------
monthly = pd.DataFrame({
    country: fetch_fred_series(series_id, START, END)
    for country, series_id in SERIES.items()
})

# ----------------------------
# 4) MAKE QUARTERLY + ANNUAL (AVERAGE OF PERIOD)
# ----------------------------
# Quarterly average of the 3 monthly values in each quarter
quarterly = monthly.resample("Q").mean()
quarterly.index = quarterly.index.to_period("Q").to_timestamp()


/tmp/ipython-input-848/3450044440.py:13: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarterly = monthly.resample("Q").mean()


In [ ]:
monthly.index = monthly.index.date
quarterly.index = quarterly.index.date
annual.index = annual.index.date

In [ ]:
print(monthly.head())
print(quarterly.head())

            United States  Canada  United Kingdom  France  Germany  Italy  \
1960-01-01            NaN    5.40            4.14     NaN     4.36    NaN   
1960-02-01            NaN    5.23            4.69     NaN     4.47    NaN   
1960-03-01            NaN    4.12            4.74     NaN     4.71    NaN   
1960-04-01            NaN    4.29            4.80     NaN     4.59    NaN   
1960-05-01            NaN    3.78            4.76     NaN     4.64    NaN   

            Japan  Spain  
1960-01-01    NaN    NaN  
1960-02-01    NaN    NaN  
1960-03-01    NaN    NaN  
1960-04-01    NaN    NaN  
1960-05-01    NaN    NaN  
            United States    Canada  United Kingdom  France   Germany  Italy  \
1960-01-01            NaN  4.916667        4.523333     NaN  4.513333    NaN   
1960-04-01            NaN  3.980000        4.863333     NaN  4.826667    NaN   
1960-07-01            NaN  3.106667        5.740000     NaN  5.543333    NaN   
1960-10-01            NaN  3.983333        5.076667    

In [ ]:
print(monthly.tail())
print(quarterly.tail())

            United States  Canada  United Kingdom    France   Germany  \
2025-09-01           4.10  2.4925            3.97  2.027273  2.027273   
2025-10-01           3.98  2.3340            3.93  2.034435  2.034435   
2025-11-01           3.86  2.1850            3.84  2.041650  2.041650   
2025-12-01           3.74  2.1640            3.75  2.045783  2.045783   
2026-01-01           3.63  2.1900            3.71  2.027727  2.027727   

               Italy    Japan     Spain  
2025-09-01  2.027273  0.81818  2.027273  
2025-10-01  2.034435  0.80909  2.034435  
2025-11-01  2.041650  0.80636  2.041650  
2025-12-01  2.045783  1.07273  2.045783  
2026-01-01  2.027727  1.12000  2.027727  
            United States    Canada  United Kingdom    France   Germany  \
2025-01-01       4.320000  2.853500            4.46  2.556836  2.556836   
2025-04-01       4.306667  2.634833            4.20  2.107691  2.107691   
2025-07-01       4.200000  2.603167            4.00  2.011572  2.011572   
2025-10-0

In [ ]:
summary = []

for country in monthly.columns:
    first_date = monthly[country].first_valid_index()
    last_date = monthly[country].last_valid_index()

    summary.append({
        "Country": country,
        "Start_Date": first_date,
        "End_Date": last_date
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

          Country  Start_Date    End_Date
0   United States  1964-06-01  2026-01-01
1          Canada  1960-01-01  2026-01-01
2  United Kingdom  1960-01-01  2026-01-01
3          France  1970-01-01  2026-01-01
4         Germany  1960-01-01  2026-01-01
5           Italy  1978-10-01  2026-01-01
6           Japan  2002-04-01  2026-01-01
7           Spain  1977-01-01  2026-01-01


In [ ]:
output_file = "G7_Spain_3Month_Rates_1960_2026.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    monthly.to_excel(writer, sheet_name="Monthly")
    quarterly.to_excel(writer, sheet_name="Quarterly")

print("Excel file saved successfully!")

Excel file saved successfully!
